In [ ]:
import inspect
import os
import subprocess
import sys
import time

from pathlib import Path
from tqdm import tqdm


from lhotse import (
    CutSet,
)


#if you need to mount your code entry point
sys.path.append("native-L2-coordinate-geometry/src")


from config import (
    DATA_PATH,
    MANIFEST_PATH,
    SANDI_CEFR_DICT,
    MODELS,
)

from build_phone_codes import (
    AveragingStrategy,    
    build_phone_codebook_timit,
    build_phone_codebook_from_alignment_tuple,
)

from utils import (
    get_cuts,
    load_lexicon,
    load_pickle,
    PhoneClass,
    save_pickle,
)

#############
#Variables
#############
#label for phone assessment
PHONE_CLASS = PhoneClass.DIPHONE
AVG_STRATEGY = AveragingStrategy.CENTER_PHONE

DEVICE='cuda'
FRAME_SHIFT = 0.02 
RANK = 16


#############
#Data
#############
#examples: either call get_cuts or use lhotse from_file

DATA_PARTS_TIMIT = ['TRAIN', 'DEV', 'TEST']
cutsets_timit = get_cuts(corpus_name='timit', partition_names=DATA_PARTS_TIMIT, manifest_dir=MANIFEST_PATH)
cuts_train_timit = cutsets_timit['TRAIN']
cuts_dev_timit = cutsets_timit['DEV']
cuts_test_timit = cutsets_timit['TEST']


sandi_cutset_path =  /path/to/sandi
cuts_train_sandi = CutSet.from_file(
    sandi_cutset_path
)


##############
#Alignments
##############
#prepa
alignment_set = {split: /path/to/alignment}
#to select a subset you can provide a index file. This will contain the cutset indices
indices = /path/to/indices





In [1]:
########
#TIMIT has its own alignment and requires a different API
########

def build_timit_codebooks(
    models,
    common_path,
    cuts_train,
    cuts_dev,
    cuts_test,
    phone_mapping,
    strategies,
    phone_classes,
    output_dir,
    ranks=(8, 16, 32, 64, 128, 256),
):

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, info in models.items():
        for layer in info["layer"]:

            print(f"\n{'=' * 70}")
            print(f"Encoder: {name} | Layer: {layer}")
            print(f"{'=' * 70}")

            feature_root = (
                DATA_PATH
                / "features"
                / name
                / "timit"
                / f"layer{layer}"
            )

            features_train = load_feature_pickles(feature_root / "train")
            features_dev = load_feature_pickles(feature_root / "dev")
            features_test = load_feature_pickles(feature_root / "test")

            for strategy in strategies:
                strategy_name = (
                    strategy.value
                    if hasattr(strategy, "value")
                    else str(strategy)
                )

                for phone_class in phone_classes:
                    if (
                        strategy == AveragingStrategy.FULL_UNIT.value
                        and phone_class == PhoneClass.MONOPHONE
                    ):
                        continue
                        
                    phone_class_name = (
                        phone_class.value
                        if hasattr(phone_class, "value")
                        else str(phone_class)
                    )

                    print(
                        f"\n{name} | layer={layer} | "
                        f"strategy={strategy_name} | "
                        f"phone_class={phone_class_name}"
                    )

                    codebooks = {}

                    for rank in ranks:
                        print(f"  Building rank={rank}")

                        codebooks[rank] = build_phone_codebook_timit(
                            cuts_train_timit=cuts_train,
                            wavlm_feat_train_timit=features_train,
                            cuts_dev_timit=cuts_dev,
                            wavlm_feat_dev_timit=features_dev,
                            cuts_test_timit=cuts_test,
                            wavlm_feat_test_timit=features_test,
                            rank=rank,
                            phone_mapping=phone_mapping,
                            phone_class=phone_class,
                            strategy=strategy,
                            feature_by_recording_id=True,
                        )

                    strategy_output_dir = (
                        output_dir
                        / name
                        / "timit"
                        / f"layer{layer}"
                        / strategy_name
                    )
                    strategy_output_dir.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    output_path = (
                        strategy_output_dir
                        / f"{phone_class_name}.pickle"
                    )

                    save_pickle(codebooks, output_path)

                    print(f"Saved: {output_path}")

In [ ]:
############
# this function call can be used for other corpora with model based alignments
###########
def build_codebooks_with_tuple_alignment(
    task,
    models,
    common_path,
    alignment_sets,
    strategies,
    phone_classes,
    output_dir,
    alignment_frame_shift: float = 0.02,
    feature_frame_shift: float = 0.02,
    ranks=(8, 16, 32, 64, 128, 256),
    utterance_mean_normalization: bool = True,
    global_mean_normalization: bool = True,
    collapse_repeats: bool = False,
    recording_ids_white_list: list = None,
):
    if not alignment_sets:
        raise ValueError("alignment_sets must not be empty")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, info in models.items():
        for layer in info["layer"]:
            print(f"\n{'=' * 70}")
            print(f"Encoder: {name} | Layer: {layer}")
            print(f"{'=' * 70}")

            feature_sets = []
            for split, alignments in alignment_sets.items():
                feature_root = (
                    DATA_PATH
                    / "features"
                    / name
                    / task
                    / f"layer{layer}"
                    / split
                )

                print(feature_root)
                features = load_feature_pickles(feature_root)
                feature_sets.append((features, alignments))

                print(f"Loaded split={split}: {len(features)} recordings")

            for strategy in strategies:
                strategy_name = (
                    strategy.value
                    if hasattr(strategy, "value")
                    else str(strategy)
                )

                for phone_class in phone_classes:
                    if (
                        strategy == AveragingStrategy.FULL_UNIT
                        and phone_class == PhoneClass.MONOPHONE
                    ):
                        continue

                    phone_class_name = (
                        phone_class.value
                        if hasattr(phone_class, "value")
                        else str(phone_class)
                    )

                    print(
                        f"\n{name} | layer={layer} | "
                        f"strategy={strategy_name} | "
                        f"phone_class={phone_class_name}"
                    )

                    strategy_output_dir = (
                        output_dir
                        / name
                        / task
                        / f"layer{layer}"
                        / strategy_name
                    )
                    strategy_output_dir.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    output_path = (
                        strategy_output_dir
                        / f"{phone_class_name}.pickle"
                    )

                    # Load existing codebooks if available.
                    if output_path.exists():
                        with open(output_path, "rb") as f:
                            codebooks = pickle.load(f)

                        print(
                            f"Loaded existing codebook: {output_path} "
                            f"(ranks={sorted(codebooks)})"
                        )
                    else:
                        codebooks = {}

                    # Only build ranks that are missing.
                    for rank in ranks:
                        if rank in codebooks:
                            print(f"  Rank={rank} already exists -> skipping")
                            continue

                        print(f"  Building missing rank={rank}")

                        codebooks[rank] = build_phone_codebook_from_alignment_tuple(
                            datasets=feature_sets,
                            rank=rank,
                            phone_class=phone_class,
                            strategy=strategy,
                            utterance_mean_normalization=(
                                utterance_mean_normalization
                            ),
                            global_mean_normalization=(
                                global_mean_normalization
                            ),
                            alignment_frame_shift=alignment_frame_shift,
                            feature_frame_shift=feature_frame_shift,
                            collapse_repeats=collapse_repeats,
                            recording_ids_white_list=recording_ids_white_list,
                        )

                    # Save the merged dictionary.
                    save_pickle(codebooks, output_path)

                    print(f"Saved: {output_path}")
                    print(f"Available ranks: {sorted(codebooks)}")





